# 05 - FISPACT activation studies

**~20 minutes.**

### The story

The transport run is done and signed off. Now the question changes: the machine has been
operated for years, it is shut down, and somebody has to go inside to replace a component.
**What is still radioactive, how much heat does it release, and which nuclides are to blame?**

That is an activation calculation, and it is FISPACT-II's job. F4Enix sits on both ends of it:
it prepares the neutron spectrum and the irradiation history that FISPACT needs, and it reads
the output back.

### What you will use

`Spectra`, `GROUP_STRUCTURES`, `Pulse`, `IrradiationScenario`, `Nuclide`, `TCF_Computer`,
`FispactOutput`, `PathwayCollection`.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
import numpy as np
import pandas as pd

from f4enix.core.constants import TIME_UNITS
from f4enix.core.egroups import GROUP_STRUCTURES
from f4enix.core.irradiation import IrradiationScenario, Nuclide, Pulse, TCF_Computer
from f4enix.core.spectra import Spectra
from f4enix.output.fispact_legacy_out import FispactOutput, PathwayCollection
from f4enix.output.mctal import Mctal

FISP = DATA / "fispact"      # the FISPACT files used by this notebook
OUT_FISP = OUT / "fispact"   # everything this notebook writes
OUT_FISP.mkdir(exist_ok=True)

## 1. The activation workflow

An activation study is a chain, and FISPACT-II is only the middle of it:

```
MCNP neutron spectrum ---> fluxes file ---> collapse.i ---> collapsed cross sections
                                       \--> condense.i ---> condensed decay data
                                                                    |
                    irradiation history ---> FISPACT-II inventory run
                                                                    |
                                                            the .out file
                                                                    |
                          shutdown dose rate  |  decay heat  |  decay pathways
```

F4Enix covers the two ends: the **preparation** of the spectrum and of the irradiation
history, and the **post-processing** of the output. The inventory input itself - the material
masses, the `DOSE`, `HALF`, `HAZARDS` keywords - is not written by F4Enix. It is normally
written by hand, or with `pypact`, which F4Enix uses under the hood to read it back.

## 2. The neutron spectrum, from the mctal

FISPACT wants one flux value per energy group. Tally 4 of the ITER_1D run is exactly that:
a cell flux binned in energy. This is the same `Mctal` object as in notebook 03, and the
`Total` rows must go before anything else happens.

In [ ]:
mctal = Mctal(DATA / "iter_1d" / "mctal")
mctal.remove_totals(4)
flux = mctal.tallydata[4]

cell = 53                                   # a cell deep inside the blanket
spectrum = flux[flux["Cells"] == cell]

print("energy groups:", len(spectrum))
spectrum.head()

175 groups is not a coincidence: the tally was set up on **VITAMIN-J-175**, one of the
group structures FISPACT understands. `GROUP_STRUCTURES` carries twenty of them, always in
**eV** and always as bin *boundaries*, so a 175-group structure has 176 numbers.

In [ ]:
ebins = GROUP_STRUCTURES["VITAMIN-J-175"]

print("some structures :", sorted(GROUP_STRUCTURES)[:6], "...")
print("boundaries      :", len(ebins), "-> groups:", len(ebins) - 1)
# the mctal reports the upper bound of each group, in MeV
print("same bins as t4 :", np.allclose(spectrum["Energy"] * 1e6, ebins[1:], rtol=1e-3))

`Spectra` expects the values **normalised to 1** - the absolute intensity comes later, from
the irradiation scenario. It will normalise for you and log a warning; doing it explicitly is
cleaner and shows what is happening.

> **Careful:** energies in eV, values normalised, and one value fewer than boundaries. Getting
> any of the three wrong raises a `ValueError` or, worse, silently shifts the whole spectrum.

In [ ]:
values = spectrum["Value"].to_numpy()
mcnp_spectrum = Spectra(ebins, values / values.sum(), name=f"ITER1D_C{cell}")

print("name   :", mcnp_spectrum.name)
print("groups :", len(mcnp_spectrum.spectra_values))
print("sum    :", mcnp_spectrum.spectra_values.sum())

## 3. Comparing with a reference spectrum

A FISPACT `fluxes` file contains only the numbers - no energies. So `from_fispact` needs you to
say which group structure they belong to. Get that wrong and the file still reads perfectly,
which is why the group structure is worth double-checking.

In [ ]:
reference = Spectra.from_fispact(GROUP_STRUCTURES["VITAMIN-J-175"], FISP / "FLUXESS1")

print("name written in the file:", reference.name)
print("groups                  :", len(reference.spectra_values))

`plot` draws one or several spectra together. Lethargy normalization is available.

In [ ]:
fig, ax = mcnp_spectrum.plot(lethargy=True, add_spectra=[reference])

## 4. The pre-processing files

Four files, four one-liners. `fluxes` is the spectrum for the collapse; `arb_flux` is the same
spectrum in the format the group-conversion routine wants; `collapse.i` and `condense.i` are
the two small FISPACT runs that prepare the collapsed cross sections and the decay data.

In [ ]:
mcnp_spectrum.to_fispact_fluxes(OUT_FISP)
mcnp_spectrum.to_arb_flux_file(OUT_FISP)
mcnp_spectrum.print_collapse_inp(OUT_FISP)
mcnp_spectrum.print_condensed_inp(OUT_FISP)

print(sorted(p.name for p in OUT_FISP.iterdir()))

In [ ]:
print((OUT_FISP / "collapse.i").read_text())

`GETXS 1 175` is the line that ties the collapse to the group structure of the flux above.

For a tungsten component the resonance self-shielding matters, and `print_collapse_inp` can add
the whole `PROBTAB` / `SSFCHOOSE` / `SSFMASS` block for you: just call it with
`tungsten=True`.

## 5. The irradiation history

A `Pulse` is a length of time at a given source intensity. A pulse with intensity `0` is a
dwell, and that is how you describe a pulsed machine. An `IrradiationScenario` is just the
ordered list of them.

In [ ]:
pulsed = IrradiationScenario(
    [Pulse(2, 2.68e18, TIME_UNITS.YEAR)] +
    [Pulse(3290, 0), Pulse(400, 5.0e19)] * 17,
    name="two_years_then_pulses",
)
pulsed.get_collapsed_table()

`get_collapsed_table` folds repeated sequences into a single row with a multiplier, which is
the only way a scenario with a few hundred pulses is readable.

Most of the time you do not type the scenario at all: it already exists in a FISPACT input, or
in a legacy `d1stime` script.

In [ ]:
scenario = IrradiationScenario.from_fispact(FISP / "inp_fispact.i")

print("pulses:", len(scenario.pulses))
scenario.get_collapsed_table()

Cooling times are given **after** shutdown, and are cumulative by default. F4Enix builds a
short label for each of them, and those labels follow the scenario everywhere.

`to_ascii` writes the whole thing to a readable `.irr` file that `from_ascii` reads back - a
much better thing to put under version control than a FISPACT input.

In [ ]:
scenario.set_cooling_times([(1, TIME_UNITS.HOUR), (1, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR)])
print("cooling labels:", scenario.cooling_labels)

scenario.to_ascii(OUT_FISP / "scenario.irr")
print((OUT_FISP / "scenario.irr").read_text())

## 6. Time correction factors

In the D1S method the decay is not simulated: the photon yield of each daughter is scaled by a
**time correction factor**, the number of atoms present at a given cooling time per unit
production rate. It depends only on the half life of the daughter and on the irradiation
history - so it can be computed here, before any transport is run.

In [ ]:
tcf = TCF_Computer()
nuclides = [Nuclide.from_formula("Co60"), Nuclide.from_formula("Mn56")]

for nuclide in nuclides:
    print(f"{nuclide.write_to_formula():6s} half life: {tcf.get_half_life(nuclide):.3e} s")

In [ ]:
factors = tcf.compute_correction_factors(scenario, nuclides, norm=1.0)

pd.DataFrame(
    factors,
    index=scenario.cooling_labels,
    columns=[n.write_to_formula() for n in nuclides],
)

Mn56 has a half life of a couple of hours: one day after shutdown it is gone. Co60 has 5.3
years and barely moves over the same range. That contrast is the whole reason cooling times are
chosen the way they are.

> **Note.** These factors are what goes into a D1S-UNED `irrad` file. F4Enix writes it with
> `IrradiationFile.from_irradiation_schedules`, which is a topic of its own.

## 7. Generating a FISPACT input

f4enix provides a light-weight wrapper of the `pypact.InputData` class for easier interface
with f4enix materials and irradiation scenarios.

In [ ]:
from f4enix.input.materials import Material
from f4enix.input.libmanager import LibManager
from f4enix.input.fispact_inp import FispactInp # The fispact input wrapper
lm = LibManager()

material = Material.from_zaids([('W', 1)], lm, '31c')

fisp_inp = FispactInp()
fisp_inp.add_material(material, style='mass', mass=1) # also 'fuel' is supported
fisp_inp.add_irradiation_scenario(scenario, norm=1)

outfile = OUT_FISP / 'fispact_input.inp'
fisp_inp.save(outfile)

print(outfile.read_text())


the input can then be completed using the underlying `pypact.InputData` object

In [ ]:
fisp_inp.inp.enableHalflifeInOutput()
fisp_inp.inp.setProjectile(1)
# ... and so on and so forth ...

## 8. Reading a FISPACT output

A legacy FISPACT-II `.out` is a few tens of MB of fixed-format text holding every time step of
the inventory. `FispactOutput` parses the inventory through `pypact` and, on top of that, pulls
out the dose, the decay heat and the reaction pathways.

You pass the labels of the cooling times: they are attached to the **last** time steps of the
file, in order.

In [ ]:
fispact_out = FispactOutput(FISP / "testSS.out", cooling_times=["1e2 s", "1e4 s"])

print("time steps in the file :", len(fispact_out.inventory_data))
print("labelled cooling times :", fispact_out.sddr["cooling time"].unique().tolist())

`sddr` and `decay_heat` are plain DataFrames, one row per isotope per cooling time, already
sorted by contribution. `isotope % dose` is the share of that isotope, `Cumulative dose sum`
the running total - so the first rows are the ones that matter.

In [ ]:
fispact_out.sddr.head()

In [ ]:
fispact_out.decay_heat.head()

## 9. Who drives the dose

`filter_by_cum_dose` cuts the table where the cumulative contribution reaches the percentage
you ask for. This is the answer to "which nuclides do I need to worry about".

In [ ]:
top_dose = fispact_out.filter_by_cum_dose(95, "1e2 s")
top_dose

Knowing *which* nuclide is only half the story - you also want to know **which reaction made
it**, because that is what you can act on in the material specification. `add_pathways=True`
explodes every isotope row into one row per production pathway.

In [ ]:
with_paths = fispact_out.filter_by_cum_dose(95, "1e2 s", add_pathways=True)
with_paths

The pathways come from the `GENERIC PATHWAYS` block of the output. They are parsed
independently by `PathwayCollection`, and each `Pathway` knows its parent, its intermediates and
its reactions. `reduce()` strips the isomeric transitions, which are not real steps, and
`get_MT()` gives the ENDF reaction number of the first reaction - the number a D1S `reac` file
needs.

In [ ]:
pathways = PathwayCollection.from_file(FISP / "testSS.out")
print("pathways found:", len(pathways.pathways))

path = pathways.pathways[13]
print("as written :", path)
print("multistep  :", path.is_multistep())
print("reduced    :", path.reduce())
print("first MT   :", path.get_MT())

In [ ]:
pathways.to_dataframe().head(10)

---

# Exercises

## Exercise 1 - does the pulsing matter?

The `pulsed` scenario of section 5 delivers its fluence in two years of steady operation
followed by seventeen short pulses. A very common shortcut is to smear all of it into a single
steady irradiation of the same length. Find out for which daughters that shortcut is legitimate.

1. compute the total fluence and the total duration of `pulsed`, and build from them an
   equivalent scenario `steady` made of **one single pulse**, carrying the same fluence over the
   same duration;
2. give both scenarios the same cooling times: 1 hour, 1 day and 30 days;
3. compute the time correction factors of `Co60` and `Mn56` for both scenarios, with `norm=1.0`;
4. build a DataFrame `ratios`, indexed by the cooling labels and with one column per nuclide,
   holding the pulsed factor divided by the steady one. Which of the two nuclides notices the
   pulses, and why?

<details>
  <summary><b>Hints</b> <- Click to expand</summary>

* `Pulse.time` is always stored in seconds, whatever unit was used to build the pulse, and
  `Pulse.intensity` is its flux - so the fluence of a pulse is the product of the two;
* `set_cooling_times([(1, TIME_UNITS.HOUR), ...])`, and the resulting `cooling_labels` are the
  index you want for `ratios`;
* `compute_correction_factors` wants the cooling times to be set already, and gives back an
  array with one row per cooling time and one column per nuclide;
* `tcf` and `nuclides` are the ones already built in section 6.

</details>

In [ ]:
# TODO 1: the fluence and the duration of the pulsed scenario
fluence = ...
duration = ...

# TODO 2: the equivalent steady scenario, one single pulse
steady = ...

# TODO 3: the same cooling times on both scenarios: 1 h, 1 d, 30 d


# TODO 4: the correction factors of both scenarios, with norm=1.0
factors_pulsed = ...
factors_steady = ...

# TODO 5: the table of the ratios, indexed by the cooling labels
ratios = ...
ratios

In [ ]:
# --- self-check, do not modify ---
fluence_pulsed = sum(p.time * p.intensity for p in pulsed.pulses)
fluence_steady = sum(p.time * p.intensity for p in steady.pulses)

assert len(steady.pulses) == 1, (
    f"the steady scenario has {len(steady.pulses)} pulses, it should have exactly one"
)
assert abs(fluence_steady / fluence_pulsed - 1) < 1e-9, (
    f"the steady scenario carries {fluence_steady:.4e} n/cm2, the pulsed one {fluence_pulsed:.4e}"
)
assert pulsed.cooling_labels == steady.cooling_labels == ["1h", "1d", "30d"], (
    f"cooling labels are {pulsed.cooling_labels} and {steady.cooling_labels}, expected "
    "['1h', '1d', '30d'] on both"
)
assert ratios.shape == (3, 2), f"the ratio table is {ratios.shape}, expected (3, 2)"
assert abs(ratios["Co60"].to_numpy() - 1).max() < 0.05, (
    f"the Co60 ratios are {ratios['Co60'].to_numpy()}, they should all be close to 1"
)
assert ratios["Mn56"].min() > 1.5, (
    f"the Mn56 ratios are {ratios['Mn56'].to_numpy()}, the pulses should push them well above 1"
)
print("Exercise 1 correct")

## Exercise 2 - dose or heat, same culprits?

The shutdown dose rate is a shielding problem, the decay heat is a cooling problem. Both come
out of the same inventory, but not necessarily out of the same nuclides. Check it on
`fispact_out`, 100 seconds after shutdown.

1. get the isotopes that make up 95% of the **dose** and those that make up 95% of the **decay
   heat**, both at the `"1e2 s"` cooling time, and call them `dose_drivers` and `heat_drivers`;
2. label the rows of the two tables with a `nuclide` column built from the element, the isotope
   and the state, then compute the two set differences `dose_only` and `heat_only` and print
   them together with the nuclides common to both lists;
3. build a table `comparison` with one row per nuclide of the **union** of the two lists,
   holding the `nuclide` label, its `isotope % dose` and its `isotope % heat`, plus a
   `heat/dose` column with the ratio of the two shares;
4. print the nuclide with the largest ratio. Why is it a cooling problem and not a shielding
   problem?

<details>
  <summary><b>Hints</b> <- Click to expand</summary>

* `filter_by_cum_dose(95, "1e2 s")` and `filter_by_cum_heating(95, "1e2 s")`; the label must be
  written exactly as it was given to `FispactOutput`;
* the state is `""` for a ground state and `"m"` for a metastable, so
  `df["element"] + df["isotope"].astype(str) + df["state"]` is a good label;
* the shares of **all** the isotopes, not only of the drivers, are in `fispact_out.sddr` and
  `fispact_out.decay_heat`: filter them on the cooling time, then keep your union of nuclides;
* `pd.merge(dose_part, heat_part, on="nuclide")` puts the two share columns side by side.

</details>

In [ ]:
label = "1e2 s"

# TODO 1: the two lists of drivers
dose_drivers = ...
heat_drivers = ...

# TODO 2: label both tables and compare the two sets of nuclides
dose_only = ...
heat_only = ...

# TODO 3: the shares of the union of the two sets, side by side
comparison = ...

# TODO 4: who has the largest heat/dose ratio?

comparison

In [ ]:
# --- self-check, do not modify ---
assert len(dose_drivers) == 3, (
    f"the 95% dose table has {len(dose_drivers)} rows, expected 3"
)
assert len(heat_drivers) == 4, (
    f"the 95% heat table has {len(heat_drivers)} rows, expected 4"
)
assert set(dose_only) == set(), f"nothing drives the dose only, but you found {dose_only}"
assert set(heat_only) == {"Co57"}, f"the heat-only nuclide should be Co57, you found {heat_only}"
assert len(comparison) == 4, f"the comparison table has {len(comparison)} rows, expected 4"
for column in ("nuclide", "heat/dose"):
    assert column in comparison.columns, (
        f"no {column!r} column in comparison, its columns are {list(comparison.columns)}"
    )
worst = comparison.sort_values("heat/dose", ascending=False).iloc[0]
assert worst["nuclide"] == "Co57", f"the largest heat/dose ratio is {worst['nuclide']}, not Co57"
assert worst["heat/dose"] > 3, (
    f"Co57 carries {worst['heat/dose']:.2f} times its dose share as heat, expected close to 3.9"
)
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| turn an MCNP spectrum into a FISPACT flux | `Spectra`, `GROUP_STRUCTURES`, `to_fispact_fluxes` |
| read a spectrum back from a `fluxes` file | `Spectra.from_fispact` |
| feed a group conversion instead | `Spectra.to_arb_flux_file` |
| compare two spectra | `Spectra.plot(lethargy=True, add_spectra=[...])` |
| write the collapse and condense inputs | `print_collapse_inp`, `print_condensed_inp` |
| self-shield a tungsten component | `print_collapse_inp(..., tungsten=True)` |
| describe an irradiation history | `Pulse`, `IrradiationScenario`, `get_collapsed_table` |
| reuse a scenario from an existing input | `IrradiationScenario.from_fispact`, `from_legacy_d1stime` |
| keep a scenario under version control | `to_ascii` / `from_ascii` |
| get the D1S time correction factors | `TCF_Computer.compute_correction_factors` |
| read a FISPACT output | `FispactOutput`, `.sddr`, `.decay_heat`, `.inventory_data` |
| find who drives the dose or the heat | `filter_by_cum_dose`, `filter_by_cum_heating` |
| get the reaction pathways | `PathwayCollection`, `Pathway.reduce`, `Pathway.get_MT` |

## Where this goes next

The pathways and the time correction factors are the two ingredients of a **D1S-UNED** shutdown
dose rate calculation: `ReactionFile.filter_by_paths` turns the first into a `reac` file and
`IrradiationFile.from_irradiation_schedules` turns the second into an `irrad` file. F4Enix does
both, and `PathwayLibrary` even ships the pathways of past F4E activation studies so you can
skip the FISPACT run altogether. Both are covered in the documentation.

<https://f4enix.readthedocs.io/en/latest/>